In [1]:
%pip -q install torchao==0.16.0
%pip install -q "transformers==4.57.1" "peft==0.17.1" "accelerate==1.11.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
from importlib.metadata import version
from decimal import Decimal
import gc
import hashlib
import json
import re
import unicodedata

import pandas as pd
import torch
import transformers
from peft import PeftModel
from PIL import Image
from transformers import AutoProcessor, GenerationConfig, Qwen3VLForConditionalGeneration, set_seed

In [4]:
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

MLLM_ROOT = Path("/content/drive/MyDrive/MLLM")
C_ROOT = MLLM_ROOT / "mordern_llm"
D_ROOT = MLLM_ROOT / "section_d"
DEVICE = torch.device("cuda:0")

E_ROOT = MLLM_ROOT / "section_e"
C_DATA = C_ROOT / "data/sroie"
PROCESSOR_DIR = C_ROOT / "outputs/qlora_run1/best_adapter"

PILOT = D_ROOT / "outputs/sft_pilot_v1"
RECIPE_PATH = PILOT / "recipe.json"
D_LOCK_PATH = D_ROOT / "data/heldout_manifest.json"
ADAPTER = PILOT / "llm_lora/best_adapter"

manifest = read_json(C_DATA / "manifest.json")
recipe = read_json(RECIPE_PATH)
CHECKPOINT_STEP = read_json(PILOT / "llm_lora/best_validation.json")["step"]

task = recipe["task"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
EVAL_BUDGET = recipe["image_kwargs"]

DTYPE = {"torch.float16": torch.float16, "torch.bfloat16": torch.bfloat16}[recipe["dtype"]]

if DTYPE == torch.bfloat16 and torch.cuda.get_device_capability(DEVICE)[0] < 8:
    raise RuntimeError("Use a GPU with native BF16 support to match the saved recipe.")

processor = AutoProcessor.from_pretrained(PROCESSOR_DIR, use_fast=False)
processor.tokenizer.padding_side = "right"
E_ROOT.mkdir(parents=True, exist_ok=True)

print("Section C:", C_ROOT)
print("Section D:", D_ROOT)
print("Processor:", PROCESSOR_DIR)
print("LoRA checkpoint step:", CHECKPOINT_STEP)
print("Dtype:", DTYPE)

Section C: /content/drive/MyDrive/MLLM/mordern_llm
Section D: /content/drive/MyDrive/MLLM/section_d
Processor: /content/drive/MyDrive/MLLM/mordern_llm/outputs/qlora_run1/best_adapter
LoRA checkpoint step: 8
Dtype: torch.float16


In [5]:
def file_sha(path):
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

def write_json(path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def lock_json(path, data):
    if path.exists():
        assert read_json(path) == data, f"Existing lock differs: {path.name}"
    else:
        write_json(path, data)
        
def find_b_image(prefix):
    folder = D_ROOT.parent / "data/images"
    matches = [
        p for p in folder.glob(f"{prefix}*")
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    ]
    return matches[0]

In [6]:
flower = find_b_image("0c1be4")
city = find_b_image("c7fa0a")
food = find_b_image("335eb4")

def choice_sample(name, image, category, question, expected, choices):
    return {
        "sample_id": f"dev:{name}",
        "split": "dev",
        "image_path": str(image),
        "task": "vqa",
        "category": category,
        "question": question,
        "expected": expected,
        "choices": choices,
        "rule": "closed_choice_v1"
    }

DEV = [
    choice_sample("flower_color", flower, "color", "What color is the flower? Answer yellow, red, white, or pink.", "yellow", ["yellow", "red", "white", "pink"]),
    choice_sample("flower_present", flower, "existence", "Is there a flower in the image? Answer only yes or no.", "yes", ["yes", "no"]),
    choice_sample("city_street_level", city, "viewpoint", "Was this photo taken at street level? Answer only yes or no.", "no", ["yes", "no"]),
    choice_sample("food_subject", food, "object", "Which is the main subject? Answer food or vehicle.", "food", ["food", "vehicle"])
]

val_by_id = {r["id"]: r for r in manifest["val"]}

for receipt_id in recipe["validation_ids"][:2]:
    record = val_by_id[receipt_id]
    path = Path(record["image_path"])
    if not path.is_file():
        path = C_DATA / "images" / path.name

    DEV.append({
        "sample_id": f"dev:receipt:{receipt_id}",
        "split": "dev",
        "image_path": str(path),
        "task": "receipt",
        "category": "extraction",
        "question": DOC_PROMPT,
        "expected": record["reference"],
        "rule": "receipt_c_v1"
    })

for sample in DEV:
    sample["image_sha256"] = file_sha(Path(sample["image_path"]))

lock_json(E_ROOT / "dev_samples_v1.json", DEV)
print("Development samples:", len(DEV))

Development samples: 6


In [7]:
def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(unicodedata.normalize("NFKC", value).casefold().split())
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?", text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")


def score_doc(answer, reference):
    raw_valid = True
    used_cleanup = False

    try:
        prediction = json.loads(answer)
    except json.JSONDecodeError:
        raw_valid = False
        prediction = None

        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", answer.strip(), flags=re.IGNORECASE | re.DOTALL)

        if fence:
            used_cleanup = True
            try:
                prediction = json.loads(fence.group(1))
            except json.JSONDecodeError:
                pass

    schema_valid = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(value is None or isinstance(value, str) for value in prediction.values())
    )

    flags = {}

    for field in FIELDS:
        correct = False

        if schema_valid:
            predicted = prediction[field]
            expected = reference[field]

            if predicted is None or expected is None:
                correct = predicted is None and expected is None
            else:
                normalized = normal(field, predicted)
                correct = (
                    normalized is not None
                    and normalized == normal(field, expected)
                )

        flags[f"{field}_correct"] = correct

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema_valid,
        "used_fence_cleanup": used_cleanup,
        **flags,
        "all_correct": all(flags.values())
    }

**Scorer and result row**


In [8]:
def normalize_choice(text):
    text = " ".join(unicodedata.normalize("NFKC", text).casefold().split())
    return text.removesuffix(".").rstrip()

def score_sample(sample, raw):
    if sample["rule"] == "receipt_c_v1":
        details = score_doc(raw, sample["expected"])
        parsed = details["prediction"]
        valid = details["schema_valid"]
        correct = details["all_correct"]
    
    elif sample["rule"] == "closed_choice_v1":
        parsed = normalize_choice(raw)
        valid = parsed in sample["choices"]
        correct = valid and parsed == normalize_choice(sample["expected"])
        details = {}
        
    else:
        raise ValueError(f"Unknown scoring rule: {sample['rule']}")

    return {
        "parsed_answer": parsed,
        "score": int(correct),
        "status": "correct" if correct else "wrong_answer" if valid else "invalid_output",
        "details": details
    }

In [9]:
def evaluate_one(sample, setting, predict, origin="live"):
    row = {
        **sample,
        "setting": setting,
        "origin": origin,
        "raw_answer": None,
        "parsed_answer": None,
        "score": 0,
        "error": None,
    }

    try:
        row["raw_answer"] = predict(sample)
    except TimeoutError as exc:
        return {**row, "status": "timeout", "error": str(exc)}
    except Exception as exc:
        return {**row, "status": "inference_error", "error": str(exc)}

    try:
        return {**row, **score_sample(sample, row["raw_answer"])}
    except Exception as exc:
        return {**row, "status": "scorer_error", "error": str(exc)}

In [10]:
def simulated_timeout(sample):
    raise TimeoutError("Simulated timeout for the runner check")

smoke = [
    evaluate_one(DEV[0], "mock", lambda s: "Yellow."),
    evaluate_one(DEV[0], "mock", lambda s: "red"),
    evaluate_one(DEV[0], "mock", lambda s: "yellow or red"),
    evaluate_one(DEV[0], "mock", simulated_timeout),
]

display(pd.DataFrame(smoke)[["raw_answer", "parsed_answer", "score", "status"]])

,raw_answer,parsed_answer,score,status
0,Yellow.,yellow,1,correct
1,red,red,0,wrong_answer
2,yellow or red,yellow or red,0,invalid_output
3,None,None,0,timeout


In [11]:
CAPS = {"vqa": 16, "receipt": task["max_new_tokens"]}
adapter_hashes = [
    file_sha(ADAPTER / "adapter_config.json"),
    file_sha(ADAPTER / "adapter_model.safetensors")
]

protocol = {
    "base_id": recipe["model_id"],
    "base_revision": recipe["revision"],
    "adapter_path": str(ADAPTER),
    "adapter_hashes": adapter_hashes,
    "adapter_checkpoint_step": CHECKPOINT_STEP,
    "dtype": str(DTYPE),
    "quantization": None,
    "attention_implementation": "sdpa",
    "processor": str(PROCESSOR_DIR),
    "processor_use_fast": False,
    "image_kwargs": EVAL_BUDGET,
    "generation": {
        "max_new_tokens_by_task": CAPS,
        "do_sample": False,
        "use_cache": True,
        "use_model_defaults": False,
        "eos_token_id": processor.tokenizer.eos_token_id,
        "pad_token_id": processor.tokenizer.pad_token_id
    },
    "seed" : 42,
    "gpu" : torch.cuda.get_device_name(DEVICE),
    "versions": {name: version(name) for name in ("torch", "transformers", "peft", "accelerate")},
    "dev_manifest_sha256": file_sha(E_ROOT / "dev_samples_v1.json"),
    "scoring_rules": ["closed_choice_v1", "receipt_c_v1"],
    "failed_requests_in_denominator": True
}

RUN_ID = hashlib.sha256(json.dumps(protocol, sort_keys=True).encode("utf-8")).hexdigest()[:12]

E_OUT = E_ROOT / "outputs" / RUN_ID
E_OUT.mkdir(parents=True, exist_ok=True)
lock_json(E_OUT / "protocol.json", protocol)

print("Run folder:", E_OUT)

Run folder: /content/drive/MyDrive/MLLM/section_e/outputs/ba2d7237c1e2


In [12]:
def load_eval_model(setting):
    if setting not in ("base", "llm_lora"):
        raise ValueError(f"Unknown evaluation setting: {setting}")

    net = Qwen3VLForConditionalGeneration.from_pretrained(
        recipe["model_id"],
        revision=recipe["revision"],
        dtype=DTYPE,
        device_map={"": 0},
        attn_implementation="sdpa"
    )
    if setting == "llm_lora":
        net = PeftModel.from_pretrained(net, ADAPTER, is_trainable=False)

    return net.eval()


@torch.no_grad()
def predict_sample(net, sample):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": sample["question"]}
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    with Image.open(sample["image_path"]) as source:
        image = source.convert("RGB")

    batch = processor(
        text=[text], images=[image], images_kwargs=EVAL_BUDGET,
        add_special_tokens=False, return_tensors="pt"
    ).to(DEVICE)

    config = GenerationConfig(
        do_sample=False,
        use_cache=True,
        max_new_tokens=CAPS[sample["task"]],
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id
    )

    with torch.autocast("cuda", dtype=DTYPE):
        output = net.generate(**batch, generation_config=config, use_model_defaults=False)

    new_ids = output[0, batch["input_ids"].shape[1]:]
    return processor.tokenizer.decode(new_ids, skip_special_tokens=True)

In [13]:
PIPELINE_FAILURES = {"inference_error", "scorer_error", "timeout"}

def run_dev_setting(setting):
    set_seed(protocol["seed"])
    net = load_eval_model(setting)
    rows = []

    try:
        for sample in DEV:
            row = evaluate_one(sample, setting, lambda s: predict_sample(net, s))
            rows.append(row)
            
            if len(rows) == 1 and row["status"] in PIPELINE_FAILURES:
                write_json(E_OUT / f"dev_{setting}_smoke_failure.json", rows)
                raise RuntimeError(f"{setting} | {sample['sample_id']} | {row['status']}: {row['error']}")
    finally:
        del net
        gc.collect()
        torch.cuda.empty_cache()

    write_json(E_OUT / f"dev_{setting}.json", rows)
    return rows


dev_rows = []
for setting in ("base", "llm_lora"):
    dev_rows.extend(run_dev_setting(setting))

df = pd.DataFrame(dev_rows)
with pd.option_context("display.max_colwidth", None):
    display(df[["setting", "sample_id", "rule", "expected", "raw_answer", "parsed_answer", "score", "status", "error"]])

assert len(dev_rows) == 2 * len(DEV)
failures = [r for r in dev_rows if r["status"] in PIPELINE_FAILURES]
if failures:
    for row in failures:
        print(f"{row['setting']} | {row['sample_id']} | {row['status']}: {row['error']}")
    raise RuntimeError(f"Development run had {len(failures)} pipeline failure(s).")

print("Development run complete:", E_OUT)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

,setting,sample_id,rule,expected,raw_answer,parsed_answer,score,status,error
0,base,dev:flower_color,closed_choice_v1,yellow,yellow,yellow,1,correct,None
1,base,dev:flower_present,closed_choice_v1,yes,yes,yes,1,correct,None
2,base,dev:city_street_level,closed_choice_v1,no,no,no,1,correct,None
3,base,dev:food_subject,closed_choice_v1,food,food,food,1,correct,None
4,base,dev:receipt:X51006311779,receipt_c_v1,"{'company': 'ANZEL ADVERTISING (M) SDN BHD', 'date': '27/03/2018', 'total': '8.50'}","{\n ""company"": ""Anzel Advertising (M) Sdn bhd"",\n ""date"": ""27/03/2018"",\n ""total"": ""8.50""\n}","{'company': 'Anzel Advertising (M) Sdn bhd', 'date': '27/03/2018', 'total': '8.50'}",1,correct,None
5,base,dev:receipt:X51006414429,receipt_c_v1,"{'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '02 APR 2018', 'total': '$8.20'}","{\n ""company"": ""UNIHAKKA INTERNATIONAL SDN BHD"",\n ""date"": ""02 Apr 2018 18:31"",\n ""total"": ""8.20""\n}","{'company': 'UNIHAKKA INTERNATIONAL SDN BHD', 'date': '02 Apr 2018 18:31', 'total': '8.20'}",0,wrong_answer,None
6,llm_lora,dev:flower_color,closed_choice_v1,yellow,yellow,yellow,1,correct,None
7,llm_lora,dev:flower_present,closed_choice_v1,yes,yes,yes,1,correct,None
8,llm_lora,dev:city_street_level,closed_choice_v1,no,no,no,1,correct,None
9,llm_lora,dev:food_subject,closed_choice_v1,food,food,food,1,correct,None


Development run complete: /content/drive/MyDrive/MLLM/section_e/outputs/ba2d7237c1e2


In [14]:
old_dir = D_ROOT / "outputs/day4_heldout"
old_lock = D_LOCK_PATH

old_records = read_json(old_lock)["test"]
old_ids = [r["id"] for r in old_records]

reference_by_id = {r["id"]: r["reference"] for r in old_records}
regression_cache = {}

for setting in ("base", "llm_lora"):
    path = old_dir / f"{setting}.json"
    if not path.exists():
        print(setting, "— cache missing")
        continue

    saved = read_json(path)
    expected_signature = {
        "recipe": file_sha(RECIPE_PATH),
        "test_lock": file_sha(old_lock),
        "weights": [] if setting == "base" else adapter_hashes
    }

    if saved["signature"] != expected_signature:
        print(setting, "— signature mismatch; skipped")
        continue

    rows = saved["rows"]
    if ([r["id"] for r in rows] != old_ids or any(r["reference"] != reference_by_id[r["id"]] for r in rows)):
        print(setting, "— sample IDs or references differ; skipped")
        continue

    regression_cache[setting] = rows

if len(regression_cache) == 2:
    write_json(E_OUT / "receipt_regression_cache.json", {
        "split": "regression",
        "origin": "section_d_day4",
        "predictions": regression_cache
    })
    print("Regression cache ready:", len(old_ids), "paired receipts")
else:
    print("No complete paired regression cache imported.")

Regression cache ready: 30 paired receipts


**Visual perception and object hallucination**

In [15]:
%pip install -q "datasets==4.4.1" "huggingface-hub>=0.34,<1"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.


In [16]:
import random
from collections import defaultdict

from datasets import Image as HFImage, load_dataset
from huggingface_hub import HfApi

In [17]:
MME = ("lmms-lab-encoder/MME", "default", "test")
POPE = ("lmms-lab-encoder/POPE", "Full", "adversarial")
CATEGORIES = ["existence", "color", "count", "position"]

D2_DATA = E_ROOT / "day2_data"
D2_DATA.mkdir(parents=True, exist_ok=True)
D2_LOCK = D2_DATA / "samples_v1.json"

def stream_rows(spec, revision, columns, filters=None):
    repo, config, split = spec
    kwargs = {"columns": columns}
    if filters is not None:
        kwargs["filters"] = filters
    return load_dataset(repo, config, split=split, revision=revision, streaming=True, **kwargs)

num_words = dict(zip("one two three four five six seven eight nine ten".split(), range(1, 11)))
number_pattern = "[0-9]+|" + "|".join(num_words)
count_pattern = re.compile(
    rf"Are there (?:exactly )?({number_pattern}) (.+?) "
    rf"in (?:this|the) (?:image|picture|photo)",
    re.IGNORECASE
)

def extract_exact_count(question):
    prompt = re.split(r"\?|\s*Please\b", question, maxsplit=1, flags=re.I)[0]
    match = count_pattern.fullmatch(prompt.strip().rstrip("."))
    if not match:
        return None
    n, objects = match.groups()
    n = n.lower()
    return (int(n) if n.isdigit() else num_words[n], objects)

In [ ]:
def choose_examples(mme_rev, pope_rev):
    rng = random.Random(42)
    groups = defaultdict(list)
    
    for r in stream_rows(MME, mme_rev, ["question_id", "category", "question", "answer"]):
        if r["category"] in CATEGORIES:
            groups[r["category"], r["question_id"]].append(r)
        
    mme_selected, direct = [], []
    for category in CATEGORIES:
        pairs = {
            qid: {r["answer"].strip().lower(): r for r in rows}
            for (cat, qid), rows in groups.items()
            if cat == category and len(rows) == 2
            and {r["answer"].strip().lower() for r in rows} == {"yes", "no"}
        }
        
        if category == "count":
            eligible = [
                qid for qid in sorted(pairs)
                if extract_exact_count(pairs[qid]["yes"]["question"]) is not None
            ]
            
            direct_ids = rng.sample(eligible, 4)
            picked = direct_ids + rng.sample(sorted(set(pairs) - set(direct_ids)), 3)

            for qid in direct_ids:
                yes_row = pairs[qid]["yes"]
                n, objects = extract_exact_count(yes_row["question"])
                direct.append((yes_row, n, objects))
        else:
            picked = rng.sample(sorted(pairs), 7)

        for qid in picked:
            mme_selected.extend([pairs[qid]["yes"], pairs[qid]["no"]])

    by_image = defaultdict(lambda: defaultdict(list))
    for r in stream_rows(POPE, pope_rev, ["question_id", "image_source", "question", "answer"]):
        by_image[r["image_source"]][r["answer"].strip().lower()].append(r)


    eligible = sorted(img for img, answers in by_image.items() if {"yes", "no"} <= answers.keys())
    pope_selected = []
    for image_id in rng.sample(eligible, 25):
        for label in ("yes", "no"):
            choices = sorted(by_image[image_id][label], key=lambda r: r["question_id"])
            pope_selected.append(rng.choice(choices))

    return mme_selected, pope_selected, direct

In [19]:
def save_selected_images(spec, revision, question_ids):
    needed = set(question_ids)
    saved = {}
    images = stream_rows(spec, revision, ["question_id", "image"], filters=[("question_id", "in", sorted(needed))]).cast_column("image", HFImage(decode=False))

    for row in images:
        qid = row["question_id"]
        if qid in saved:
            continue
        raw = row["image"]["bytes"]
        if not raw:
            raise ValueError(f"Missing embedded image bytes: {qid}")
        
        sha = hashlib.sha256(raw).hexdigest()
        path = D2_DATA / f"{sha}.img"
        
        if not path.exists():
            path.write_bytes(raw)
        saved[qid] = {"image_path": str(path), "image_sha256": sha}
        if saved.keys() == needed:
            break

    return saved

In [ ]:
if not D2_LOCK.is_file():
    api = HfApi()
    mme_rev = api.dataset_info(MME[0]).sha
    pope_rev = api.dataset_info(POPE[0]).sha
    mme_rows, pope_rows, direct = choose_examples(mme_rev, pope_rev)

    mme_images = save_selected_images(MME, mme_rev, [r["question_id"] for r in mme_rows])
    
    first_by_image = {}
    for r in pope_rows:
        first_by_image.setdefault(r["image_source"], r["question_id"])
    pope_images = save_selected_images(POPE, pope_rev, first_by_image.values())

    samples = []
    for r in mme_rows:
        qid, category = r["question_id"], r["category"]
        label = r["answer"].strip().lower()
        pair_id = f"mme:{category}:{qid}"
        samples.append({
            **mme_images[qid], "sample_id": f"{pair_id}:{label}",
            "split": "eval", "benchmark": "MME", "task": "vqa",
            "category": category, "pair_id": pair_id,
            "source_question_id": qid,
            "question": r["question"], "expected": label,
            "choices": ["yes", "no"], "rule": "closed_choice_v1"
        })

    for r in pope_rows:
        qid = r["question_id"]
        samples.append({
            **pope_images[first_by_image[r["image_source"]]],
            "sample_id": f"pope:adversarial:{qid}",
            "split": "eval", "benchmark": "POPE", "task": "vqa",
            "category": "adversarial", "source_question_id": qid,
            "source_image_id": r["image_source"],
            "question": r["question"].strip() + " Answer only yes or no.",
            "expected": r["answer"].strip().lower(),
            "choices": ["yes", "no"], "rule": "closed_choice_v1"
        })

    for r, n, objects in direct:
        qid = r["question_id"]
        samples.append({
            **mme_images[qid], "sample_id": f"direct_count:{qid}",
            "split": "eval", "benchmark": "Direct count", "task": "vqa",
            "category": "count", "source_question_id": qid,
            "question": f"How many {objects} are in the image? Answer with digits only.",
            "expected": n, "rule": "integer_v1"
        })

    lock_json(D2_LOCK, {
        "seed": 42,
        "sources": {"MME": [*MME, mme_rev], "POPE": [*POPE, pope_rev]},
        "samples": samples
    })

D2 = read_json(D2_LOCK)["samples"]
assets = {s["image_path"]: s["image_sha256"] for s in D2}

display(pd.DataFrame(D2).groupby(["benchmark", "category"]).size())
display(pd.DataFrame(D2).query("benchmark == 'Direct count'")[["question", "expected"]])

README.md:   0%|          | 0.00/493 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

benchmark     category   
Direct count  count           4
MME           color          14
              count          14
              existence      14
              position       14
POPE          adversarial    50
dtype: int64

,question,expected
106,How many horses are in the image? Answer with ...,2
107,How many bananas are in the image? Answer with...,2
108,How many laptops are in the image? Answer with...,3
109,How many people are in the image? Answer with ...,4


**BASE and LORA**

In [ ]:
D2_OUT = E_OUT / "day2_perception"
D2_OUT.mkdir(exist_ok=True)
lock_json(D2_OUT / "protocol.json", {
    "day1_protocol_sha256": file_sha(E_OUT / "protocol.json"),
    "samples_sha256": file_sha(D2_LOCK),
    "rules": ["closed_choice_v1", "integer_v1"],
    "pipeline_failures_in_accuracy_denominator": True,
})

def run_day2(setting):
    path = D2_OUT / f"{setting}.json"
    rows = read_json(path) if path.exists() else []
    assert [r["sample_id"] for r in rows] == [s["sample_id"] for s in D2[:len(rows)]]
    if len(rows) == len(D2):
        return rows

    set_seed(protocol["seed"])
    net = load_eval_model(setting)
    start = len(rows)

    try:
        for sample in D2[start:]:
            row = evaluate_one(sample, setting, lambda s: predict_sample(net, s))
            if len(rows) == start and row["status"] in {"inference_error", "scorer_error", "timeout"}:
                raise RuntimeError(f"{setting} | {sample['sample_id']} | {row['error']}")

            rows.append(row)
            if len(rows) % 10 == 0 or len(rows) == len(D2):
                tmp = path.with_suffix(".tmp")
                write_json(tmp, rows)
                tmp.replace(path)
                print(f"{setting}: {len(rows)}/{len(D2)} saved", flush=True)
    finally:
        del net
        gc.collect()
        torch.cuda.empty_cache()

    return rows

day2_rows = run_day2("base") + run_day2("llm_lora")
day2_df = pd.DataFrame(day2_rows)
print("Total results:", len(day2_df))

In [ ]:
mme = day2_df[day2_df["benchmark"] == "MME"]
mme_questions = mme.groupby(["setting", "category"]).agg(
    questions=("score", "size"),
    accuracy=("score", "mean"),
    invalid_rate=("status", lambda s: s.eq("invalid_output").mean()),
    pipeline_errors=("status", lambda s: s.isin({"timeout", "inference_error", "scorer_error"}).sum()),
)

pairs = mme.groupby(["setting", "category", "pair_id"])["score"].agg(
    questions="size", both_correct="min",
)
assert pairs["questions"].eq(2).all()
mme_pairs = pairs.groupby(["setting", "category"]).agg(
    pairs=("both_correct", "size"),
    pair_accuracy=("both_correct", "mean"),
)
mme_summary = mme_questions.join(mme_pairs)
display(mme_summary.round(3))

pope = day2_df[day2_df["benchmark"] == "POPE"]
pope_summary = []

for setting, g in pope.groupby("setting"):
    pos, neg = g["expected"].eq("yes"), g["expected"].eq("no")
    assert pos.sum() == neg.sum() == 25
    valid = g["status"].isin(["correct", "wrong_answer"])
    yes = g["parsed_answer"].eq("yes") & valid

    fp = int((neg & yes).sum())
    tp = int((pos & yes).sum())
    pope_summary.append({
        "setting": setting,
        "n": len(g),
        "accuracy": g["score"].mean(),
        "FP / negatives": f"{fp}/{neg.sum()}",
        "FPR": fp / neg.sum(),
        "TP / positives": f"{tp}/{pos.sum()}",
        "positive_recall": tp / pos.sum(),
        "invalid_rate": g["status"].eq("invalid_output").mean(),
        "invalid_on_no": int((neg & g["status"].eq("invalid_output")).sum()),
        "pipeline_errors": int(g["status"].isin(
            {"timeout", "inference_error", "scorer_error"}
        ).sum()),
    })

pope_summary = pd.DataFrame(pope_summary)
display(pope_summary.round(3))

count = day2_df[day2_df["benchmark"] == "Direct count"]
count_summary = count.groupby("setting")["score"].agg(n="size", correct="sum", accuracy="mean")
display(count_summary)
display(count[["setting", "question", "expected", "raw_answer", "status"]])

mme_summary.to_csv(D2_OUT / "mme_summary.csv")
pope_summary.to_csv(D2_OUT / "pope_summary.csv", index=False)
count_summary.to_csv(D2_OUT / "direct_count_summary.csv")